In [ ]:
# =================================================================
# 3. MATRICES ESTRATIFICADAS (Modelo de reflexión por Impedancia)
# =================================================================

def T_interface(nu, Gamma):
    """Matriz de interfaz geométrica. nu=+1 (entrada), nu=-1 (salida)"""
    mat = np.array([
        [1.0, nu * Gamma],
        [nu * Gamma, 1.0]
        ], dtype=complex)
    return (1.0 / (1.0 - nu * Gamma)) * mat

def T_cell_stratified_impedance(w, Ms_profile, dx, use_BVMSW = False):
    """
    Construye la matriz de la celda unidad multiplicando las N rebanadas
    usando la analogía de línea de transmisión (salto de impedancia).
    """
    N_slices = len(Ms_profile)
    T_total = np.eye(2, dtype=complex) # Matriz Identidad inicial
    curr_nu = 1

    for j in range(N_slices):
        Ms_curr = Ms_profile[j]

        if use_BVMSW:
            k_curr = k_wave_bvmsw(w, Ms_curr)
            kp_curr = k_prime_bvmsw(w, Ms_curr)
        else:
            k_curr = k_wave(w, Ms_curr)
            kp_curr = k_prime(w, Ms_curr)

        # 1. Matriz de propagación a través de la rebanada actual

        # Acumulación de fase y decaimiento de amplitud
        exp_minus = np.exp(-(1j * k_curr - kp_curr) * dx)
        exp_plus  = np.exp((1j * k_curr - kp_curr) * dx)

        T_prop = np.array([
            [exp_minus, 0.0],
            [0.0, exp_plus]
        ], dtype=complex)

        # 2. Matriz de interfaz
        # El índice circular (j+1) % N_slices conecta el final con el inicio de la celda
        Ms_next = Ms_profile[(j + 1) % N_slices]
        k_next = k_wave(w, Ms_next)

        # Cálculo del coeficiente de reflexión local (desacople de vectores de onda)
        # Se añade un pequeño epsilon para evitar división por cero accidental
        Gamma = (k_curr - k_next) / (k_curr + k_next)

        T_int_plus = T_interface(curr_nu, Gamma)
        T_int_minus = T_interface(-1, Gamma)

        # 3. Multiplicación de matrices

        T_total = T_total @ T_prop @ T_int_plus
        curr_nu = -curr_nu

    return T_total

def transmission_power(f_GHz, Ms_profile, dx, use_BVMSW = False):
    """Coeficiente de potencia transmitida Pw para el cristal de N periodos"""
    w = 2 * np.pi * f_GHz
    tc = T_cell_stratified_impedance(w, Ms_profile, dx, use_BVMSW)
    # Matriz elevada a la potencia N
    T_total = np.linalg.matrix_power(tc, Nperiods)
    T11 = T_total[0, 0]
    return 1.0 / (np.abs(T11)**2)  # P_tr = 1 / |T_11|^2 = 1 / |T_22|^2
